# Fight model training - MoViNet-A0 on crops of the people

Run every cell top to bottom (or **Save Version -> Save & Run All**).

Before running, in the right-hand panel:
- **Accelerator: GPU T4 x2**
- **Internet: on** (downloads MoViNet's starting weights - Kinetics-600, Apache 2.0 - and the packages)
- **Add Input: "RWF-2000"** and **"Real Life Violence Situations"** (RLVS)

What happens:
1. find the videos and label them from their folders (Fight / NonFight, Violence / NonViolence)
2. turn every video into 1-second clips the way the live app will see them: 8 frames at 8 fps,
   cropped to the people (found with **yolo11n-pose**, used as-is), resized to 172 x 172
3. fine-tune **MoViNet-A0** (stream version) - the only model trained here
4. test it on videos it never saw (RWF-2000's own `val` folder)

Result: **`/kaggle/working/fight_movinet.pt`** - download it from the **Output** tab and put it
in the project's `models/` folder.

In [ ]:
!pip install -q ultralytics==8.4.136 imageio-ffmpeg git+https://github.com/roggerfq/MoViNet-pytorch.git

In [ ]:
# --- 1. Settings, and find the videos and their labels ----------------------
import os, re, json, random, time, pickle, subprocess, sys
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn

NOTEBOOK_STARTED = time.time()
INPUT_DIR = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))     # saved as the notebook's output
CACHE_DIR = Path(os.environ.get("CACHE_DIR", "/tmp/fight_cache"))  # big temporary files - not saved
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# These must match the live app, or the model learns from a different picture
# than it will see: frames scaled down to 640 wide, 8 frames sampled at 8 fps
# (one second), each resized to 172 x 172 - what MoViNet-A0 takes.
WIDTH = 640
RATE = 8.0
WINDOW = 8
STRIDE = 4               # a new clip every half second
SIZE = 172
MAX_WINDOWS = 12         # at most this many clips from one video, so long videos don't dominate
MAX_SECONDS = 60         # read at most this much of any one video
CLASSES = ["normal", "fight"]
SEED = 0
EPOCHS = int(os.environ.get("EPOCHS", "12"))
BATCH = int(os.environ.get("BATCH", "32"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "4" if sys.platform.startswith("linux") else "0"))
TIME_BUDGET_HOURS = 9.0  # stop training early rather than hit Kaggle's 12-hour limit and lose everything

VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv", ".mpg", ".mpeg", ".webm"}
NORMAL_FOLDERS = {"nonfight", "non-fight", "non_fight", "nonviolence", "non-violence", "non_violence",
                  "noviolence", "normal"}
FIGHT_FOLDERS = {"fight", "fights", "violence", "violent"}
TEST_FOLDERS = {"val", "test", "validation"}       # RWF-2000 keeps its own held-out set in val/


def label_of(parts):
    """1 fight, 0 normal, None if the folders don't say."""
    lowered = [p.lower() for p in parts]
    if any(p in NORMAL_FOLDERS for p in lowered):
        return 0
    if any(p in FIGHT_FOLDERS for p in lowered):
        return 1
    return None


items, seen_before = [], set()
left_out = {"folder doesn't say fight or not": 0, "duplicate copy": 0}
for video in sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in VIDEO_EXTS):
    parts = video.relative_to(INPUT_DIR).parts
    label = label_of(parts[:-1])
    if label is None:
        left_out["folder doesn't say fight or not"] += 1
        continue
    key = (video.stem.lower(), video.stat().st_size)
    if key in seen_before:
        left_out["duplicate copy"] += 1         # the same video twice would leak into the test
        continue
    seen_before.add(key)
    items.append({"id": re.sub(r"[^A-Za-z0-9]+", "_", str(video.relative_to(INPUT_DIR))),
                  "video": video, "label": label,
                  "held_out": any(p.lower() in TEST_FOLDERS for p in parts[:-1]),
                  "source": "/".join(parts[:min(3, len(parts) - 1)])})

# Split by VIDEO, never by clip. RWF-2000's own val/ folder is the test set, so the
# result compares with published numbers; 10% of the rest checks progress while
# training. With no val/ folder anywhere, 15% is held out for the test instead.
rng = random.Random(SEED)
has_test = any(it["held_out"] for it in items)
for label in (0, 1):
    group = [it for it in items if it["label"] == label and not it["held_out"]]
    rng.shuffle(group)
    n_test = 0 if has_test else int(len(group) * 0.15)
    n_val = int(len(group) * 0.10)
    for k, it in enumerate(group):
        it["split"] = "test" if k < n_test else ("val" if k < n_test + n_val else "train")
for it in items:
    if it["held_out"]:
        it["split"] = "test"

print(f"videos: {len(items)}  (fight: {sum(it['label'] for it in items)}, "
      f"normal: {sum(1 - it['label'] for it in items)})")
print("left out:", left_out)
by_source = {}
for it in items:
    by_source.setdefault(it["source"], [0, 0])[it["label"]] += 1
for source, (normal, fight) in sorted(by_source.items()):
    print(f"  {source:60} fight {fight:5}  normal {normal:5}")
for s in ("train", "val", "test"):
    group = [it for it in items if it["split"] == s]
    print(f"{s:5}: {len(group):5} videos  ({sum(it['label'] for it in group)} fight)")
if not items or all(it["label"] == 1 for it in items) or all(it["label"] == 0 for it in items):
    print("\nWARNING: need both fight and normal videos. Stop here and check the inputs.")

In [ ]:
# --- 2. Turn every video into 1-second clips of the people ------------------
# In a separate worker process, like the fall notebook: if a video (or the GPU)
# crashes it, only the worker dies - the notebook notes which video, skips it and
# carries on. Videos are read without their audio track at all.
WORKER = r'''
import faulthandler, json, os, pickle, sys
faulthandler.enable()                      # if it crashes, print where
from pathlib import Path
import cv2
import imageio_ffmpeg
import numpy as np
from ultralytics import YOLO

tasks_file, progress_file, device, weights = sys.argv[1:5]
width, rate, window, stride, size, max_windows, max_seconds = (
    int(sys.argv[5]), float(sys.argv[6]), int(sys.argv[7]), int(sys.argv[8]),
    int(sys.argv[9]), int(sys.argv[10]), float(sys.argv[11]))
person_model = YOLO(weights)


def people_crop(shape, boxes, grow=1.25, min_side=96):
    """The square of the picture holding everyone seen during the clip, with a
    margin. Nobody seen: the whole picture. The live app must crop exactly this way."""
    h, w = shape[:2]
    if not boxes:
        return 0, 0, w, h
    b = np.array(boxes)
    x1, y1, x2, y2 = b[:, 0].min(), b[:, 1].min(), b[:, 2].max(), b[:, 3].max()
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = min(max(max(x2 - x1, y2 - y1) * grow, min_side), h, w)
    x0 = int(round(min(max(cx - side / 2, 0), w - side)))
    y0 = int(round(min(max(cy - side / 2, 0), h - side)))
    s = int(round(side))
    return x0, y0, x0 + s, y0 + s


def frames_of(video):
    """The picture only, at `rate` fps, scaled down to `width` like the live app."""
    reader = imageio_ffmpeg.read_frames(video, pix_fmt="bgr24",
                                        output_params=["-map", "0:v:0", "-vf", f"fps={rate}"])
    meta = next(reader)
    w, h = meta["size"]
    for k, raw in enumerate(reader):
        if k >= max_seconds * rate:
            break
        frame = np.frombuffer(raw, np.uint8).reshape(h, w, 3)
        if w > width:
            frame = cv2.resize(frame, (width, round(h * width / w)), interpolation=cv2.INTER_AREA)
        else:
            frame = frame.copy()
        yield frame


def prepare(video):
    frames = list(frames_of(video))
    if len(frames) < window:
        raise ValueError(f"only {len(frames)} frames")
    boxes = []
    for i in range(0, len(frames), 32):
        for r in person_model(frames[i:i + 32], verbose=False, device=device):
            boxes.append(r.boxes.xyxy.cpu().numpy().tolist() if r.boxes is not None else [])
    starts = list(range(0, len(frames) - window + 1, stride))
    if len(starts) > max_windows:
        starts = [starts[round(k * (len(starts) - 1) / (max_windows - 1))] for k in range(max_windows)]
    clips, people = [], []
    for s in starts:
        x0, y0, x1, y1 = people_crop(frames[s].shape, [b for f in range(s, s + window) for b in boxes[f]])
        clip = []
        for f in range(s, s + window):
            crop = cv2.resize(frames[f][y0:y1, x0:x1], (size, size), interpolation=cv2.INTER_AREA)
            clip.append(cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes())
        clips.append(clip)
        people.append(max(len(boxes[f]) for f in range(s, s + window)))
    return {"starts": starts, "clips": clips, "people": people, "frames": len(frames)}


for k, (video, out) in enumerate(json.load(open(tasks_file))):
    Path(progress_file).write_text(str(k))
    try:
        result = prepare(video)
    except Exception as error:             # unreadable video: report it, carry on
        print("failed", k, repr(error)[:300], flush=True)
        continue
    part = out + ".part"
    with open(part, "wb") as f:
        pickle.dump(result, f)
    os.replace(part, out)                  # never leaves a half-written file behind
    print("done", k, flush=True)
'''

CLIPS = CACHE_DIR / "clips"
CLIPS.mkdir(exist_ok=True)
WORKER_PATH = CACHE_DIR / "prepare_worker.py"
WORKER_PATH.write_text(WORKER)
uploaded = sorted(INPUT_DIR.rglob("yolo11n-pose.pt"))
WEIGHTS = str(uploaded[0]) if uploaded else "yolo11n-pose.pt"   # your copy, else the official one
device = os.environ.get("POSE_DEVICE") or ("0" if torch.cuda.is_available() else "cpu")
print("person finder:", uploaded[0] if uploaded else "official yolo11n-pose", "  device:", device)

counter = {"done": 0}
started = time.time()


def out_path(item):
    return CLIPS / (item["id"] + ".pkl")


def run_worker(batch, on):
    """Runs the worker over `batch` on device `on`. Returns (the index it crashed
    at, or None), [(index, reason)] for videos it couldn't read, its last error lines."""
    tasks, progress, errors = CACHE_DIR / "tasks.json", CACHE_DIR / "progress.txt", WORK_DIR / "worker_errors.txt"
    tasks.write_text(json.dumps([(str(it["video"]), str(out_path(it))) for it in batch]))
    progress.unlink(missing_ok=True)
    unreadable = []
    with open(errors, "w") as err:
        proc = subprocess.Popen(
            [sys.executable, str(WORKER_PATH), str(tasks), str(progress), on, WEIGHTS, str(WIDTH), str(RATE),
             str(WINDOW), str(STRIDE), str(SIZE), str(MAX_WINDOWS), str(MAX_SECONDS)],
            stdout=subprocess.PIPE, stderr=err, text=True, cwd=CACHE_DIR)
        for line in proc.stdout:
            word, _, rest = line.strip().partition(" ")
            if word == "done":
                counter["done"] += 1
                if counter["done"] % 100 == 0:
                    print(f"{counter['done']}/{len(items)} videos  {time.time() - started:.0f}s", flush=True)
            elif word == "failed":
                k, _, why = rest.partition(" ")
                unreadable.append((int(k), why))
        proc.wait()
    crashed = None
    if proc.returncode != 0:
        crashed = int(progress.read_text()) if progress.exists() else 0
    return crashed, unreadable, errors.read_text(errors="ignore").strip().splitlines()[-12:]


left_out_videos = {}
pending = [it for it in items if not out_path(it).exists()]
counter["done"] = len(items) - len(pending)
crashes_in_a_row = 0
while pending:
    crashed, unreadable, last_lines = run_worker(pending, device)
    for k, why in unreadable:
        left_out_videos[pending[k]["id"]] = why
    if crashed is None:
        break
    if crashed > 0:
        crashes_in_a_row = 0                  # it got through some videos first
    video = pending[crashed]
    print(f"\nthe worker crashed on {video['video'].name} (device {device}). Its last messages:")
    print("  " + "\n  ".join(last_lines))
    if device != "cpu":
        print("trying that video on the CPU, to tell a GPU problem from a bad video...")
        again, unreadable_again, _ = run_worker([video], "cpu")
        if again is None and not unreadable_again:
            device = "cpu"
            print("it worked on the CPU, so the GPU was the problem - carrying on on the CPU (slower)\n")
            crashes_in_a_row = 0
            pending = pending[crashed + 1:]
            continue
    left_out_videos[video["id"]] = "crashed the worker"
    crashes_in_a_row += 1
    if crashes_in_a_row >= 5:
        raise RuntimeError("5 videos in a row crashed - stopping here. Send the messages above.")
    pending = pending[crashed + 1:]

if left_out_videos:
    print(f"\nleft out {len(left_out_videos)} videos that could not be read:")
    for name, why in list(left_out_videos.items())[:10]:
        print("  ", name, "-", why[:150])
items = [it for it in items if it["id"] not in left_out_videos and out_path(it).exists()]

samples = {"train": [], "val": [], "test": []}
for it in items:
    with open(out_path(it), "rb") as f:
        prepared = pickle.load(f)
    it["clip_count"] = len(prepared["clips"])
    for clip, people in zip(prepared["clips"], prepared["people"]):
        samples[it["split"]].append((clip, it["label"], it["id"], people))
print(f"\nclips ready from {len(items)} videos  ({time.time() - started:.0f}s)")
for s, group in samples.items():
    print(f"{s:5}: {len(group):6} clips  ({sum(g[1] for g in group)} fight)")

# What the model will actually see: the middle frame of a few clips of each kind.
tiles = []
for label in (1, 0):
    group = [g for g in samples["train"] if g[1] == label]
    picked = group[::max(1, len(group) // 6)][:6]
    for clip, _, _, _ in picked:
        tile = cv2.imdecode(np.frombuffer(clip[WINDOW // 2], np.uint8), cv2.IMREAD_COLOR)
        cv2.putText(tile, CLASSES[label], (4, 16), 0, 0.5, (0, 0, 255), 1)
        tiles.append(tile)
if tiles:
    while len(tiles) % 6:
        tiles.append(np.zeros_like(tiles[0]))
    cv2.imwrite(str(WORK_DIR / "what_the_model_sees.jpg"),
                np.vstack([np.hstack(tiles[i:i + 6]) for i in range(0, len(tiles), 6)]))
    print("examples saved: what_the_model_sees.jpg")

In [ ]:
# --- 3. Fine-tune MoViNet-A0 - the only model trained here ------------------
from movinets import MoViNet
from movinets.config import _C

MOVINET_WEIGHTS_DIR = os.environ.get("MOVINET_WEIGHTS_DIR", str(CACHE_DIR / "movinet_weights"))


class FightMoViNet(nn.Module):
    """MoViNet-A0, stream version, with a new last layer: [normal, fight].
    Input: batch x 3 (RGB, 0..1) x 8 frames x 172 x 172.

    pretrained=True starts from the Kinetics-600 weights (downloaded). To load a
    trained file later, build it with pretrained=False - the settings below make
    it the identical network either way."""

    def __init__(self, pretrained):
        super().__init__()
        self.backbone = MoViNet(_C.MODEL.MoViNetA0, causal=True, pretrained=pretrained,
                                num_classes=600, conv_type="2plus1d", tf_like=True,
                                model_dir=MOVINET_WEIGHTS_DIR if pretrained else None)
        self.backbone.classifier[3] = nn.Identity()          # 2048 features out, not 600 actions
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2048, len(CLASSES)))

    def forward(self, x):
        # The stream version carries memory from one call to the next. Each clip
        # is judged on its own, so start clean every time - the live app does too.
        self.backbone.clean_activation_buffers()
        return self.head(self.backbone(x))


def augment(clip):
    """More variety from the same clips: mirror, a slightly tighter crop, lighter
    or darker, sometimes grey (night cameras). The same change for all 8 frames."""
    if random.random() < 0.5:
        clip = clip[:, :, ::-1]
    side = int(SIZE * random.uniform(0.8, 1.0))
    y, x = random.randint(0, SIZE - side), random.randint(0, SIZE - side)
    if side < SIZE:
        clip = np.stack([cv2.resize(f[y:y + side, x:x + side], (SIZE, SIZE), interpolation=cv2.INTER_AREA)
                         for f in clip])
    clip = clip.astype(np.float32) * random.uniform(0.75, 1.25) + random.uniform(-20, 20)
    if random.random() < 0.1:
        clip = np.repeat(clip.mean(axis=-1, keepdims=True), 3, axis=-1)
    return np.clip(clip, 0, 255)


class Clips(torch.utils.data.Dataset):
    def __init__(self, group, train):
        self.group, self.train = group, train

    def __len__(self):
        return len(self.group)

    def __getitem__(self, i):
        jpegs, label = self.group[i][0], self.group[i][1]
        clip = np.stack([cv2.imdecode(np.frombuffer(j, np.uint8), cv2.IMREAD_COLOR) for j in jpegs])
        clip = clip[..., ::-1]                                # BGR -> RGB, as the live classifier does
        if self.train:
            clip = augment(clip)
        x = torch.from_numpy(np.ascontiguousarray(clip, dtype=np.float32) / 255.0).permute(3, 0, 1, 2)
        return x, label


def loader(group, train):
    return torch.utils.data.DataLoader(Clips(group, train), batch_size=BATCH, shuffle=train,
                                       num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                                       drop_last=train and len(group) > BATCH)


def predict(model, group):
    """Fight probability for each clip."""
    model.eval()
    out = []
    with torch.no_grad():
        for x, _ in loader(group, train=False):
            with torch.autocast(device_type=amp_device, dtype=torch.float16, enabled=use_amp):
                logits = model(x.to(train_device, non_blocking=True))
            out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu())
    return torch.cat(out).numpy() if out else np.zeros(0)


def scores(prob, truth, threshold=0.5):
    """precision: of the alarms, how many were fights. recall: of the fights, how many were caught."""
    hit = prob >= threshold
    tp = int((hit & (truth == 1)).sum())
    fp = int((hit & (truth == 0)).sum())
    fn = int((~hit & (truth == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


random.seed(SEED)
torch.manual_seed(SEED)
# not "mps" on a Mac: MoViNet uses an operation Apple's GPU doesn't have
train_device = os.environ.get("TRAIN_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
use_amp = train_device == "cuda"                    # half precision on the GPU: about twice as fast
amp_device = "cuda" if use_amp else "cpu"
print("training on:", train_device)
model = FightMoViNet(pretrained=True).to(train_device)
optimiser = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-4},     # already knows movement - nudge it
    {"params": model.head.parameters(), "lr": 1e-3},         # brand new - learn fast
], weight_decay=1e-4)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, EPOCHS)
scaler = torch.amp.GradScaler(enabled=use_amp)
loss_fn = nn.CrossEntropyLoss()

CONFIG = {
    "model": "FightMoViNet (MoViNet-A0 stream, causal, conv_type 2plus1d, tf_like) + Linear(2048, 2)",
    "build": "FightMoViNet(pretrained=False), then load_state_dict",
    "classes": CLASSES, "width": WIDTH, "rate": RATE, "window": WINDOW, "size": SIZE,
    "input": "RGB, 0..1, batch x 3 x 8 x 172 x 172; clean_activation_buffers() before every call",
    "crop": "square around all yolo11n-pose person boxes in the 8 frames, x1.25, at least 96 px, "
            "at most the frame's shorter side; nobody seen -> whole frame; cv2.INTER_AREA to 172",
    "trained_on": sorted(by_source), "seed": SEED, "epochs": EPOCHS,
}
MODEL_PATH = WORK_DIR / "fight_movinet.pt"
check = "val" if samples["val"] else "train"
check_truth = np.array([g[1] for g in samples[check]])
train_loader = loader(samples["train"], train=True)
best = -1.0
for epoch in range(1, EPOCHS + 1):
    epoch_started = time.time()
    model.train()
    total, count = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(train_device, non_blocking=True), y.to(train_device, non_blocking=True)
        with torch.autocast(device_type=amp_device, dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(x), y)
        optimiser.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        total += loss.item() * len(y)
        count += len(y)
    schedule.step()
    precision, recall, f1 = scores(predict(model, samples[check]), check_truth)
    note = ""
    if f1 > best:
        best = f1
        torch.save({"state_dict": model.state_dict(), "config": CONFIG}, MODEL_PATH)
        note = "  <- best so far, saved"
    print(f"epoch {epoch:2}/{EPOCHS}  loss {total / max(count, 1):.3f}  {check}: precision {precision:.2f} "
          f"recall {recall:.2f} f1 {f1:.2f}  ({time.time() - epoch_started:.0f}s){note}", flush=True)
    hours = (time.time() - NOTEBOOK_STARTED) / 3600
    if hours > TIME_BUDGET_HOURS:
        print(f"stopping early: {hours:.1f} hours used - keeping the best model saved so far")
        break

In [ ]:
# --- 4. Test on videos the model never saw ----------------------------------
model.load_state_dict(torch.load(MODEL_PATH, map_location=train_device, weights_only=True)["state_dict"])
test = samples["test"]
prob = predict(model, test)
truth = np.array([g[1] for g in test])
precision, recall, f1 = scores(prob, truth)
print(f"test clips (1 second each): {len(test)}  precision {precision:.2f}  recall {recall:.2f}  f1 {f1:.2f}\n")

# Per video, as published RWF-2000 results are counted: a video is called a fight
# if its clips average over the line.
per_video = {}
for p, (_, label, video_id, _) in zip(prob, test):
    per_video.setdefault(video_id, [label, []])[1].append(float(p))
results = {"test_clips": {"count": len(test), "precision": precision, "recall": recall, "f1": f1}}
for threshold in (0.5, 0.7, 0.9):
    right = caught = fights = false_alarms = normals = 0
    for label, ps in per_video.values():
        called = float(np.mean(ps)) >= threshold
        right += called == bool(label)
        if label:
            fights += 1
            caught += called
        else:
            normals += 1
            false_alarms += called
    accuracy = right / max(len(per_video), 1)
    results[f"threshold_{threshold}"] = {"video_accuracy": accuracy, "fights_caught": caught, "fights": fights,
                                         "false_alarms": false_alarms, "normal_videos": normals}
    print(f"threshold {threshold}: accuracy {accuracy:.1%} over {len(per_video)} videos - "
          f"caught {caught}/{fights} fights, {false_alarms}/{normals} normal videos wrongly called a fight")

(WORK_DIR / "fight_results.json").write_text(json.dumps(results, indent=2))
print(f"\nmodel saved: {MODEL_PATH}   ({(time.time() - NOTEBOOK_STARTED) / 60:.0f} min in all)")

## Download

Open the notebook's **Output** tab, download **`fight_movinet.pt`**, and put it in the project's `models/` folder.
`fight_results.json` has the test numbers above; `what_the_model_sees.jpg` shows example clips.